# 🦢 Swin UNETR on the SAME BraTS subset — the comparison arm

**Goal:** train MONAI's **Swin UNETR** on *exactly* the same data split as the nnU-Net arm
(notebook 02) and evaluate with *exactly* the same metrics — a controlled comparison.

**📎 Required Kaggle Datasets:**
1. [`awsaf49/brats-2021-task-data`](https://www.kaggle.com/datasets/awsaf49/brats-2021-task-data)
2. **The output of notebook 02** (Save Version on it first, then Add Input → your notebook output)
   — we need `brats_subset_split.json` from it.

**⚙️ Settings:** GPU T4 x2 · Internet On.

### Fair-comparison protocol (this is the research core — keep it identical!)
| Knob | nnU-Net arm | Swin UNETR arm |
|---|---|---|
| Cases | first 50 sorted, 80/20 split | **same file** `brats_subset_split.json` |
| Preprocessing | 1 mm iso resample, per-channel foreground z-score (nnU-Net CT-independent scheme) | **matched manually below** |
| Budget | `nnUNetTrainer_100epochs`, fold 0 | 100 epochs, fold 0 split, similar patch volume |
| Metrics | `src/metrics.py` (Dice, HD95-mm) | **identical code path** |

> ⚠️ Honest limitation to state in your writeup: nnU-Net additionally auto-tunes patch/batch
> size, augmentation and architecture per dataset; we give Swin UNETR sensible fixed defaults
> instead of an equally tuned setup. The comparison is *budget-matched*, not *tuning-matched*.

## 0 · Setup & data

In [ ]:
!pip install -q -U "monai[nibabel]>=1.3"
import os, glob, json, pathlib, numpy as np, pandas as pd
SPLIT_FILES = glob.glob("/kaggle/input/*/brats_subset_split.json") + glob.glob("/kaggle/working/**/brats_subset_split.json", recursive=True)
assert SPLIT_FILES, "Attach the OUTPUT of notebook 02 as a Kaggle Dataset (it contains brats_subset_split.json)!"
split = json.load(open(SPLIT_FILES[0]))
print("split file:", SPLIT_FILES[0])
print("train:", len(split["train"]), " val:", len(split["val"]))

In [ ]:
!git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo 2>/dev/null || true
import sys; sys.path.insert(0, "/kaggle/working/repo")

In [ ]:
# map BraTS21_XXXXX -> original case folder (notebook 02 recorded id_map)
cands = glob.glob("/kaggle/input/*brats*")
TRAIN_DIRS = glob.glob(f"{cands[0]}/**/RSNA_ASNR_MICCAI_BraTS_2021_TrainingData", recursive=True) if cands else []
TRAIN_DIR = TRAIN_DIRS[0] if TRAIN_DIRS else cands[0]
id_map = split["id_map"]   # {"BraTS21_00001": "BraTS2021_00xxx", ...}
orig_of = {k: os.path.join(TRAIN_DIR, v) for k, v in id_map.items()}
missing = [k for k, v in orig_of.items() if not os.path.isdir(v)]
assert not missing, f"{len(missing)} case folders not found — is the BraTS dataset attached? e.g. {missing[:3]} (looked in {TRAIN_DIR})"
print("✅ all", len(orig_of), "case folders resolved")

## 1 · Dataset class — preprocessing matched to nnU-Net's BraTS scheme

BraTS scans are already skull-stripped, co-registered and 1 mm isotropic, so resampling is a
no-op (we assert it). Intensity handling follows nnU-Net's default for MRI-like data:
**clip to foreground percentiles [0.5, 99.5], then z-score with foreground mean/std** — per channel.

In [ ]:
import torch, nibabel as nib
from collections import OrderedDict
from torch.utils.data import Dataset

MODS = ("flair", "t1", "t1ce", "t2")
# BraTS {0,1,2,4} -> consecutive {0,1,2,3}; a lookup table beats np.vectorize here
_REMAP_LUT = np.array([0, 1, 2, 0, 3], dtype=np.int8)

def load_case(new_id):
    d = orig_of[new_id]
    imgs = []
    for m in MODS:
        hits = [f for f in glob.glob(os.path.join(d, f"*{m}.nii.gz")) if "seg" not in os.path.basename(f).lower()]
        imgs.append(nib.load(hits[0]))
    seg = nib.load(glob.glob(os.path.join(d, "*seg.nii.gz"))[0])
    X = np.stack([np.asarray(im.dataobj, dtype=np.float32) for im in imgs])   # (4, D, H, W)
    y = _REMAP_LUT[np.clip(np.asarray(seg.dataobj).astype(np.int16), 0, 4)]
    return X, y, imgs[0].affine, new_id

def normalize(x):
    fg = x[x > 0]
    if fg.size == 0: return x
    lo, hi = np.percentile(fg, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    fg = x[x > 0]
    return (x - fg.mean()) / (fg.std() + 1e-8)

class BraTSSubset(Dataset):
    """Random 96^3 crops for training.

    The cache is bounded on purpose: a BraTS volume is ~36 MB as float32 and two
    DataLoader workers would otherwise keep every case in RAM (13 GB on Kaggle).
    """
    def __init__(self, ids, augment=False, cache_size=4):
        self.ids, self.augment = list(ids), augment
        self.cache = OrderedDict()
        self.cache_size = cache_size
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        cid = self.ids[i]
        if cid not in self.cache:
            X, y, aff, _ = load_case(cid)
            assert X.shape[1:] == (240, 240, 155), f"unexpected BraTS shape {X.shape} — data not 1mm iso?"
            X = np.stack([normalize(c) for c in X])
            self.cache[cid] = (X.astype(np.float32), y)
            if len(self.cache) > self.cache_size:
                self.cache.popitem(last=False)
        X, y = self.cache[cid]
        X, y = X.copy(), y.copy()
        if self.augment:
            if np.random.rand() < 0.5: X, y = X[..., ::-1].copy(), y[..., ::-1].copy()
            if np.random.rand() < 0.5: X, y = X[:, :, ::-1].copy(), y[:, :, ::-1].copy()
            X = X + np.random.normal(0, 0.01, X.shape).astype(np.float32)
        # random crop 96^3 (same volume ballpark as nnU-Net's auto-planned patch)
        D, H, W = X.shape[1:]
        ps = 96
        d0 = np.random.randint(0, D - ps); h0 = np.random.randint(0, H - ps); w0 = np.random.randint(0, W - ps)
        Xc = X[:, d0:d0+ps, h0:h0+ps, w0:w0+ps]
        yc = y[d0:d0+ps, h0:h0+ps, w0:w0+ps]
        # ensure the crop contains foreground ~80% of the time (else re-center on a lesion)
        if yc.max() == 0 and np.random.rand() < 0.8:
            fg = np.argwhere(y > 0)
            if len(fg):
                c = fg[np.random.randint(len(fg))]
                d0 = int(np.clip(c[0]-ps//2, 0, D-ps)); h0 = int(np.clip(c[1]-ps//2, 0, H-ps)); w0 = int(np.clip(c[2]-ps//2, 0, W-ps))
                Xc = X[:, d0:d0+ps, h0:h0+ps, w0:w0+ps]; yc = y[d0:d0+ps, h0:h0+ps, w0:w0+ps]
        return torch.from_numpy(Xc), torch.from_numpy(yc.astype(np.int64))

train_ds = BraTSSubset(split["train"], augment=True)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
print(f"train batches/epoch: {len(train_dl)}   val cases: {len(split['val'])}")

## 2 · Model — MONAI Swin UNETR (pretrained on BTCV, 4 in-channels, 4 classes)

In [ ]:
from monai.networks.nets import SwinUNETR
device = torch.device("cuda")
model = SwinUNETR(img_size=(96, 96, 96), in_channels=4, out_channels=4,
                  feature_size=24, use_v2=True).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"SwinUNETR params: {n_params/1e6:.1f}M")

## 3 · Training — 100 epochs, DiceCE loss, cosine LR

**Budget honesty:** nnU-Net's "100 epochs" means 100 × 250 iterations ≈ 25k optimizer
steps. One Swin epoch here is a single pass over the 40 training volumes (~40 steps), so the
two arms are matched on *epochs*, not on *iterations*. I state that in the results instead of
pretending the budgets are identical.

In [ ]:
from monai.inferers import sliding_window_inference

def dice_per_class(pred, gt, n_classes=4):
    out = []
    for c in range(1, n_classes):
        p, g = pred == c, gt == c
        denom = p.sum() + g.sum()
        out.append(1.0 if denom == 0 else float(2 * (p & g).sum() / denom))
    return out

@torch.no_grad()
def validate_full(model, ids, roi=(96, 96, 96)):
    """Dice on the FULL held-out volumes — not on random 96^3 crops.

    Validating on random crops (the easy shortcut) reports a number that depends
    on where the crop happened to land. This is the honest version, and it is the
    only one worth putting next to nnU-Net's validation Dice.
    """
    model.eval()
    scores = []
    for cid in ids:
        X, y, _, _ = load_case(cid)
        Xn = np.stack([normalize(c) for c in X]).astype(np.float32)
        logits = sliding_window_inference(torch.from_numpy(Xn[None]).to(device),
                                          roi_size=roi, sw_batch_size=1, predictor=model, overlap=0.5)
        pred = torch.argmax(torch.softmax(logits, 1), 1)[0].cpu().numpy()
        scores.append(float(np.mean(dice_per_class(pred, y))))
    model.train()
    return float(np.mean(scores))

In [ ]:
%%time
from monai.losses import DiceCELoss
import time

EPOCHS = 100
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
loss_fn = DiceCELoss(include_background=False, softmax=True, to_onehot_y=True)

best_dice, history = -1, []
for epoch in range(EPOCHS):
    model.train(); t0 = time.time(); ep_loss = 0.0
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad(set_to_none=True); loss.backward()
        optimizer.step(); ep_loss += loss.item()
    scheduler.step()
    ep_loss /= len(train_dl)

    if (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:      # full-volume val every 10 epochs
        val = validate_full(model, split["val"])
        history.append({"epoch": epoch+1, "train_loss": round(ep_loss, 4),
                        "val_dice": round(val, 4), "lr": scheduler.get_last_lr()[0]})
        print(f"epoch {epoch+1:3d} | loss {ep_loss:.4f} | val_dice {val:.4f} | {time.time()-t0:.0f}s/epoch")
        if val > best_dice:
            best_dice = val
            torch.save({"model": model.state_dict(), "epoch": epoch+1,
                        "val_dice": val, "history": history}, "/kaggle/working/swinunetr_best.pt")
    else:
        history.append({"epoch": epoch+1, "train_loss": round(ep_loss, 4)})
print("✅ best val dice:", best_dice)

## 4 · Inference on validation cases → NIfTI + evaluation (identical code path)

In [ ]:
%%time
import nibabel as nib
# weights_only=False: the checkpoint also carries the plain-python training history
ckpt = torch.load("/kaggle/working/swinunetr_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"]); model.eval()
out_dir = "/kaggle/working/predictions/brats_swin_val"; os.makedirs(out_dir, exist_ok=True)
t_inf = []
with torch.no_grad():
    for cid in split["val"]:
        X, y, aff, _ = load_case(cid)
        Xn = np.stack([normalize(c) for c in X]).astype(np.float32)
        t0 = time.time()
        logits = sliding_window_inference(torch.from_numpy(Xn[None]).to(device), roi_size=(96,96,96),
                                          sw_batch_size=2, predictor=model, overlap=0.5)
        pred = torch.argmax(torch.softmax(logits, 1), 1)[0].cpu().numpy().astype(np.uint8)
        t_inf.append(time.time() - t0)
        nib.save(nib.Nifti1Image(pred, aff), os.path.join(out_dir, f"{cid}.nii.gz"))
peak = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f"mean inference time/volume: {np.mean(t_inf):.1f}s   peak GPU mem: {peak:.0f} MB")

In [ ]:
%%time
import sys; sys.path.insert(0, "/kaggle/working/repo" if os.path.isdir("/kaggle/working/repo") else ".")
# metrics module: clone repo OR it's attached with notebook 02's output; fallback: pip-installed copy
try:
    from src.metrics import evaluate_folders
except ImportError:
    !git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo
    sys.path.insert(0, "/kaggle/working/repo")
    from src.metrics import evaluate_folders

# GT folder in the SAME remapped label space: build it from the raw labels
gt_dir = "/kaggle/working/gt_brats_remapped"; os.makedirs(gt_dir, exist_ok=True)
for cid in split["val"]:
    if not os.path.exists(f"{gt_dir}/{cid}.nii.gz"):
        _, y, aff, _ = load_case(cid)
        nib.save(nib.Nifti1Image(y.astype(np.uint8), aff), f"{gt_dir}/{cid}.nii.gz")
report = evaluate_folders(out_dir, gt_dir, labels=(1, 2, 3))
for k, v in report.mean.items():
    print(f"{k:>10} ->", {kk: round(vv, 4) for kk, vv in v.items()})
report.save(pathlib.Path("/kaggle/working/results/brats_swinunetr"))

In [ ]:
# model stats for the efficiency part of the comparison
import json
stats = {"model": "SwinUNETR (MONAI, feature_size=24, v2)", "n_params_M": round(n_params/1e6, 2),
         "epochs": EPOCHS, "patch": 96, "batch": 1,
         "mean_inference_s_per_volume": round(float(np.mean(t_inf)), 1),
         "peak_gpu_mem_MB": round(float(peak), 0),
         "best_val_dice": round(float(ckpt["val_dice"]), 4)}
json.dump(stats, open("/kaggle/working/results/brats_swinunetr/model_stats.json", "w"), indent=2)
pd.DataFrame(ckpt["history"]).to_csv("/kaggle/working/results/brats_swinunetr/training_history.csv", index=False)
print(json.dumps(stats, indent=2))

## ✅ What this notebook produced
- `predictions/brats_swin_val/` — Swin UNETR predictions on the shared val split
- `results/brats_swinunetr/` — per-case CSV + summary.json (same format as the nnU-Net arm)
- `swinunetr_best.pt` — checkpoint (needed for XAI in notebook 05)
- `training_history.csv` + `model_stats.json` — learning curve & efficiency numbers

**Next:** notebook 04 merges both arms into the comparison table + figures.